In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    FunctionTransformer,
    OneHotEncoder,
    OrdinalEncoder,
    RobustScaler,
)

In [2]:
class Pre_Processor:

    def __init__(self, model_type="linear"):
        self.model_type = model_type
        self._label_data()
        self._build_pipelines()

    def _label_data(self):
        self.numerical_cols = ["age", "avg_glucose_level", "bmi"]
        self.nominal_cols = ["gender", "work_type", "smoking_status"]
        self.binary_cols = [
            "hypertension",
            "heart_disease",
            "ever_married",
            "Residence_type",
        ]
        self.ordinal_cols = []

        self.all_cat_cols = self.nominal_cols + self.binary_cols

    def _build_pipelines(self):
        steps = [("imputer", SimpleImputer(strategy="median"))]
        if self.model_type == "linear":
            steps.extend(
                [
                    (
                        "log_transform",
                        FunctionTransformer(np.log1p, validate=False),
                    ),
                    ("scaler", RobustScaler()),
                ]
            )
        self.numerical_pipeline = Pipeline(steps=steps)

        self.cat_imputer = SimpleImputer(strategy="most_frequent")
        self.nominal_encoder = OneHotEncoder(
            handle_unknown="ignore", sparse_output=False
        )
        self.binary_encoder = OrdinalEncoder()

    def _remove_duplicates(self, df):
        return df.drop_duplicates().reset_index(drop=True)

    def _domain_clean(self, df):
        valid_age = (df["age"] >= 0) & (df["age"] <= 120)
        valid_bmi = (df["bmi"] > 0) | df["bmi"].isna()
        valid_gender = df["gender"].isin(["Male", "Female"])

        clean_mask = valid_age & valid_bmi & valid_gender
        return df[clean_mask].reset_index(drop=True)

    def _clean_categorical_strings(self, df):
        df_clean = df.copy()
        for col in self.all_cat_cols:
            if col in df_clean.columns:
                df_clean[col] = (
                    df_clean[col]
                    .astype(str)
                    .str.strip()
                    .replace(
                        {
                            "NAN": np.nan,
                            "NaN": np.nan,
                            "Unknown": np.nan,
                            "": np.nan,
                        }
                    )
                )
        return df_clean

    def fit(self, X_train, y_train=None):
        """Fit all transformers strictly on training data."""
        if y_train is not None:
            train_split = pd.concat([X_train, y_train], axis=1)
            train_split = self._remove_duplicates(train_split)
            train_split = self._domain_clean(train_split)
            X_clean = train_split.drop(columns=[y_train.name])
        else:
            X_clean = X_train.copy()

        X_clean = self._clean_categorical_strings(X_clean)

        self.numerical_pipeline.fit(X_clean[self.numerical_cols])

        cat_imputed_arr = self.cat_imputer.fit_transform(
            X_clean[self.all_cat_cols]
        )
        cat_imputed_df = pd.DataFrame(
            cat_imputed_arr, columns=self.all_cat_cols, index=X_clean.index
        )

        self.nominal_encoder.fit(cat_imputed_df[self.nominal_cols])
        self.binary_encoder.fit(cat_imputed_df[self.binary_cols])

        return self

    def transform(self, X, y=None):
        """Transform any set (X_train, X_val, X_test) using fitted parameters."""
        if y is not None:
            split_df = pd.concat([X, y], axis=1)
            split_df = self._remove_duplicates(split_df)
            split_df = self._domain_clean(split_df)
            X_clean = split_df.drop(columns=[y.name])
            y_clean = split_df[y.name].reset_index(drop=True)
        else:
            X_clean = X.copy()
            y_clean = None

        X_clean = self._clean_categorical_strings(X_clean)

        numerical_processed = self.numerical_pipeline.transform(
            X_clean[self.numerical_cols]
        )

        cat_imputed_arr = self.cat_imputer.transform(
            X_clean[self.all_cat_cols]
        )
        cat_imputed_df = pd.DataFrame(
            cat_imputed_arr, columns=self.all_cat_cols, index=X_clean.index
        )

        nominal_processed = self.nominal_encoder.transform(
            cat_imputed_df[self.nominal_cols]
        )
        binary_processed = self.binary_encoder.transform(
            cat_imputed_df[self.binary_cols]
        )

        X_processed_arr = np.hstack(
            [numerical_processed, nominal_processed, binary_processed]
        )

        nominal_feature_names = list(
            self.nominal_encoder.get_feature_names_out(self.nominal_cols)
        )
        feature_names = (
            self.numerical_cols + nominal_feature_names + self.binary_cols
        )

        X_processed = pd.DataFrame(X_processed_arr, columns=feature_names)

        if y_clean is not None:
            return X_processed, y_clean
        return X_processed

    def fit_transform(self, X_train, y_train=None):
        """Standard scikit-learn shortcut to fit and transform in one call."""
        return self.fit(X_train, y_train).transform(X_train, y_train)